# 08 — Generate Delta CSVs from Local Projection Pickles

Replaces the Google Colab S3-extraction step (`08_colab_extract.ipynb`).
The `data/climate_projections/` folder already contains xarray DataArrays
(shape: 12 months × 5 lat × 5 lon) extracted from NEX-GDDP-CMIP6.

This notebook spatially averages those pickles over the Big Island bounding box,
computes the delta (future − historical) per month, and writes the CSV files
that `08_climate_projections.ipynb` cell 6 expects.

**Output:** `data/climate_projections/{model_slug}_{scenario}_{horizon}.csv`  
**Columns:** `month, delta_tas, delta_tasmin, delta_tasmax, ratio_pr`

After running this notebook, continue in `08_climate_projections.ipynb` from **cell 6**.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

PROJ_DIR  = '../data/climate_projections'

# Models present in PROJ_DIR (add others if pickles appear later)
MODELS    = ['ACCESS-CM2', 'MPI-ESM1-2-HR']
SCENARIOS = ['ssp245', 'ssp585']
HORIZONS  = ['2035', '2045']
VARS      = ['tas', 'tasmin', 'tasmax', 'pr']

print('Config OK')
print('PROJ_DIR:', os.path.abspath(PROJ_DIR))

Config OK
PROJ_DIR: /home/simonhans/coding/GrapeExpectations/JavaScript/data/climate_projections


In [2]:
def load_spatial_mean(model, scenario_label, var, horizon_label):
    """
    Load a DataArray pickle and return a pd.Series(12,) indexed 1–12
    by averaging over all lat/lon grid points (skipna=True).
    Returns None if the file does not exist.
    """
    fname = f'{model}_{scenario_label}_{var}_{horizon_label}.pkl'
    path  = os.path.join(PROJ_DIR, fname)
    if not os.path.exists(path):
        return None
    obj = pd.read_pickle(path)
    # Spatial mean over every dim except 'month'
    spatial_dims = [d for d in obj.dims if d != 'month']
    sm = obj.mean(dim=spatial_dims, skipna=True)
    return pd.Series(sm.values, index=obj.coords['month'].values)


def compute_delta_csv(model, scenario, horizon):
    """
    Build a 12-row delta CSV for one model/scenario/horizon.
    Returns a DataFrame or None if essential data is missing.
    Temperatures are in Kelvin — deltas are identical to °C deltas.
    """
    # --- load historical ---
    hist = {v: load_spatial_mean(model, 'historical', v, 'hist') for v in VARS}

    # Approximate tas from tasmin/tasmax if direct tas pkl is absent
    if hist['tas'] is None and hist['tasmin'] is not None and hist['tasmax'] is not None:
        hist['tas'] = (hist['tasmin'] + hist['tasmax']) / 2

    if hist['tas'] is None:
        print(f'  SKIP {model}: no historical temperature data')
        return None

    # --- load future ---
    futr = {v: load_spatial_mean(model, scenario, v, horizon) for v in VARS}

    if futr['tas'] is None and futr['tasmin'] is not None and futr['tasmax'] is not None:
        futr['tas'] = (futr['tasmin'] + futr['tasmax']) / 2

    if futr['tas'] is None:
        print(f'  SKIP {model}/{scenario}/{horizon}: no future temperature data')
        return None

    rows = []
    for month in range(1, 13):

        def g(d, var):
            s = d.get(var)
            return float(s[month]) if s is not None and month in s.index else np.nan

        h_tas    = g(hist, 'tas');    f_tas    = g(futr, 'tas')
        h_tasmin = g(hist, 'tasmin'); f_tasmin = g(futr, 'tasmin')
        h_tasmax = g(hist, 'tasmax'); f_tasmax = g(futr, 'tasmax')
        h_pr     = g(hist, 'pr');     f_pr     = g(futr, 'pr')

        # If tasmin/tasmax missing in future, reconstruct from tas delta
        if np.isnan(f_tasmin) and not np.isnan(h_tasmin):
            f_tasmin = h_tasmin + (f_tas - h_tas)
        if np.isnan(f_tasmax) and not np.isnan(h_tasmax):
            f_tasmax = h_tasmax + (f_tas - h_tas)

        # Precip ratio; fall back to 1.0 (no change) if pr data is absent
        if np.isnan(h_pr) or np.isnan(f_pr) or h_pr < 1e-15:
            ratio_pr = 1.0
        else:
            ratio_pr = float(np.clip(f_pr / h_pr, 0.1, 5.0))

        rows.append({
            'month':        month,
            'delta_tas':    f_tas    - h_tas,
            'delta_tasmin': f_tasmin - h_tasmin,
            'delta_tasmax': f_tasmax - h_tasmax,
            'ratio_pr':     ratio_pr,
        })

    return pd.DataFrame(rows)


print('Helper functions defined.')

Helper functions defined.


In [3]:
generated = []
skipped   = []

for model in MODELS:
    model_slug = model.replace('-', '_')
    print(f'\n=== {model} ===')

    for scenario in SCENARIOS:
        for horizon in HORIZONS:
            out_csv = os.path.join(PROJ_DIR, f'{model_slug}_{scenario}_{horizon}.csv')

            if os.path.exists(out_csv):
                print(f'  cached: {os.path.basename(out_csv)}')
                generated.append(out_csv)
                continue

            df = compute_delta_csv(model, scenario, horizon)
            if df is None:
                skipped.append(f'{model}/{scenario}/{horizon}')
                continue

            df.to_csv(out_csv, index=False)
            avg_dt = df['delta_tas'].mean()
            avg_pr = df['ratio_pr'].mean()
            print(f'  saved: {os.path.basename(out_csv)}')
            print(f'         delta_tas avg={avg_dt:+.2f} °C  |  ratio_pr avg={avg_pr:.3f}')
            generated.append(out_csv)

print(f'\nDone. {len(generated)} CSVs ready, {len(skipped)} skipped.')
if skipped:
    print('Skipped (missing pkl data):')
    for s in skipped:
        print(f'  {s}')


=== ACCESS-CM2 ===
  cached: ACCESS_CM2_ssp245_2035.csv
  cached: ACCESS_CM2_ssp245_2045.csv
  cached: ACCESS_CM2_ssp585_2035.csv
  cached: ACCESS_CM2_ssp585_2045.csv

=== MPI-ESM1-2-HR ===
  cached: MPI_ESM1_2_HR_ssp245_2035.csv
  cached: MPI_ESM1_2_HR_ssp245_2045.csv
  SKIP MPI-ESM1-2-HR/ssp585/2035: no future temperature data
  SKIP MPI-ESM1-2-HR/ssp585/2045: no future temperature data

Done. 6 CSVs ready, 2 skipped.
Skipped (missing pkl data):
  MPI-ESM1-2-HR/ssp585/2035
  MPI-ESM1-2-HR/ssp585/2045


In [4]:
# ── Sanity check: print one CSV to confirm format ─────────────────────────────
sample = os.path.join(PROJ_DIR, 'ACCESS_CM2_ssp245_2035.csv')
if os.path.exists(sample):
    df = pd.read_csv(sample)
    print(f'{os.path.basename(sample)}:')
    print(df.to_string(index=False, float_format='{:+.3f}'.format))
else:
    print('Sample CSV not found — check skipped list above.')

print()
print('Next step: open 08_climate_projections.ipynb and run from cell 6 onward.')

ACCESS_CM2_ssp245_2035.csv:
 month  delta_tas  delta_tasmin  delta_tasmax  ratio_pr
     1     +1.038        +1.093        +0.983    +1.246
     2     +0.898        +0.949        +0.846    +0.992
     3     +0.963        +0.958        +0.969    +0.809
     4     +1.119        +1.103        +1.135    +1.086
     5     +1.176        +1.174        +1.178    +1.173
     6     +1.333        +1.334        +1.332    +1.004
     7     +1.278        +1.273        +1.283    +0.932
     8     +1.278        +1.247        +1.309    +0.980
     9     +1.180        +1.187        +1.173    +1.378
    10     +1.489        +1.502        +1.476    +1.076
    11     +1.477        +1.494        +1.460    +1.010
    12     +1.291        +1.300        +1.281    +0.892

Next step: open 08_climate_projections.ipynb and run from cell 6 onward.
